# 09.03 -- Token Streaming and Incremental Decoding

**Module:** 09 -- LLM Inference Engineering  
**Notebook:** 3 of 5  
**Prerequisites:** 09.01 (KV Cache), 09.02 (Dynamic Batching)

---

## Learning Objectives

1. Understand why streaming improves perceived latency even without changing throughput
2. Implement a token streaming pipeline using Python generators and asyncio
3. Build server-sent events (SSE) streaming compatible with the OpenAI API format
4. Measure and model the relationship between TTFT, TPOT, and perceived quality
5. Implement stopping criteria, token budgets, and stream cancellation

---

## 1. Why Streaming Matters

Without streaming, the user sees nothing until the entire response is generated. For a 500-token response at 30 tokens/second, that is a 16-second blank screen -- an unusable experience.

With streaming, the first token appears in ~100ms (the TTFT) and subsequent tokens appear at the generation rate. The user's perceived latency drops from 16 seconds to 0.1 seconds, even though the total generation time is identical.

```
Without streaming:              |========= 16 seconds =========| first word visible
With streaming (TTFT=100ms):    | 100ms | word word word word word...
                                  ^first token appears here
```

Streaming is not about going faster -- it is about **showing partial results as they are produced**. This is the standard for all modern LLM APIs (OpenAI, Anthropic, etc.).

In [ ]:
# Environment setup.
#
# asyncio is the Python standard library for asynchronous programming.
# We use it to implement non-blocking token streaming, which is how real
# production APIs deliver tokens without blocking other requests.
#
# The transformers library provides TextIteratorStreamer, a callback-based
# mechanism for streaming tokens from HuggingFace generation loops.
# We implement our own streaming primitives first to understand the mechanics,
# then show how to use the library implementation.

import asyncio
import time
import queue
import threading
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import Iterator, AsyncIterator, Optional, List
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print()
print("Key streaming concepts:")
print("  TTFT  = Time-to-First-Token  (first byte latency)")
print("  TPOT  = Time-per-Output-Token (streaming speed)")
print("  E2E   = End-to-end latency    (TTFT + TPOT * n_tokens)")

## 2. Implementing Token Streaming From Scratch

In [ ]:
# Python generator-based token streaming.
#
# A generator function is the most natural Python primitive for implementing
# token streaming. Instead of returning a complete response, the streaming
# generator yields one token at a time as it is produced.
#
# The caller iterates over the generator with a for loop, receiving and
# processing each token immediately rather than waiting for the full response.
#
# This is how HuggingFace's TextIteratorStreamer works internally, and it is
# the pattern used by the Anthropic SDK's streaming client.
#
# The decode_stream generator below simulates the timing behaviour of a real
# LLM: a fixed prefill cost followed by per-token generation. We inject
# realistic timing via time.sleep() to simulate GPU compute time.

def decode_stream(
    prompt:          str,
    tokens_to_generate: int  = 50,
    prefill_ms:      float   = 150.0,   # time to process the prompt
    decode_ms:       float   = 33.0,    # time per generated token (30 tok/s)
    vocab:           List[str] = None,
) -> Iterator[str]:
    """
    Generator that yields tokens one at a time, simulating streaming LLM output.

    In a real implementation, this generator wraps the model's autoregressive
    loop and yields each token immediately after sampling rather than waiting
    for the full sequence to complete.
    """
    if vocab is None:
        # Simple placeholder vocabulary for demonstration
        vocab = ['the', 'a', 'model', 'generates', 'tokens', 'one', 'at', 'a',
                 'time', 'for', 'streaming', 'output', 'using', 'autoregressive',
                 'decoding', 'with', 'attention', 'to', 'context', 'and']

    rng = np.random.default_rng(sum(ord(c) for c in prompt))

    # Prefill: process the prompt tokens (no output yet)
    time.sleep(prefill_ms / 1000.0)

    # Decode: yield one token per step
    for i in range(tokens_to_generate):
        time.sleep(decode_ms / 1000.0)
        token = rng.choice(vocab)
        # Add a period at the end to simulate sentence completion
        if i == tokens_to_generate - 1:
            token = token + '.'
        yield token


# Demonstrate the streaming interface
print("Streaming demo (with real timing):")
print("-" * 50)
t_start     = time.time()
first_token  = None
tokens_seen  = 0
text_so_far  = []

for token in decode_stream("Explain gradient descent.", tokens_to_generate=12,
                            prefill_ms=150, decode_ms=30):
    now = time.time()
    if first_token is None:
        first_token = now
        print(f"  TTFT: {(now - t_start)*1000:.0f}ms -- first token: '{token}'")
    text_so_far.append(token)
    tokens_seen += 1
    print(f"  t={now-t_start:.2f}s  token {tokens_seen:>2}: '{token}'")

e2e = time.time() - t_start
tpot = (e2e - (first_token - t_start)) / tokens_seen
print()
print(f"TTFT:  {(first_token - t_start)*1000:.0f}ms")
print(f"TPOT:  {tpot*1000:.0f}ms/token  ({1/tpot:.1f} tokens/s)")
print(f"E2E:   {e2e*1000:.0f}ms")

In [ ]:
# Server-Sent Events (SSE) streaming format.
#
# All production LLM APIs (OpenAI, Anthropic, Google) use Server-Sent Events
# over HTTP to deliver token streams to clients. SSE is a simple text protocol:
# each event is prefixed with 'data: ' and ended with two newlines.
#
# The OpenAI streaming format specifically wraps each token in a JSON object:
#   data: {"id":"...", "object":"chat.completion.chunk",
#           "choices":[{"delta":{"content":"token"}, "index":0}]}
#
# The stream ends with a sentinel:
#   data: [DONE]
#
# Understanding this format is essential for building LLM-powered applications
# that consume streamed responses efficiently.

import json

def format_sse_chunk(
    token:      str,
    model_id:   str = 'gpt-example',
    request_id: str = 'req-001',
    finish_reason: Optional[str] = None,
) -> str:
    """
    Format a single token as an OpenAI-compatible SSE chunk.

    This is the exact wire format sent by the OpenAI /v1/chat/completions
    endpoint when stream=True. Building a compatible format allows easy
    integration with any client library that supports OpenAI streaming.
    """
    chunk = {
        'id': request_id,
        'object': 'chat.completion.chunk',
        'model': model_id,
        'choices': [{
            'index': 0,
            'delta': {'content': token} if finish_reason is None else {},
            'finish_reason': finish_reason,
        }]
    }
    return f'data: {json.dumps(chunk)}\n\n'


def format_sse_done() -> str:
    """The terminal SSE event signalling the end of the stream."""    return 'data: [DONE]\n\n'


def streaming_response(prompt: str, n_tokens: int = 8) -> Iterator[str]:
    """
    A generator that yields SSE-formatted chunks, suitable for an HTTP response.

    In a real server (FastAPI, Flask), this generator would be the body
    of a StreamingResponse. The client reads chunks and updates the UI
    incrementally as each arrives.
    """
    req_id = f'req-{abs(hash(prompt)):x}'[:12]
    for i, token in enumerate(decode_stream(prompt, n_tokens, prefill_ms=50, decode_ms=20)):
        is_last = (i == n_tokens - 1)
        yield format_sse_chunk(token, request_id=req_id,
                               finish_reason='stop' if is_last else None)
    yield format_sse_done()


# Demonstrate the SSE wire format
print("SSE wire format demonstration (first 3 chunks + done):")
print("-" * 60)
for i, chunk in enumerate(streaming_response("What is attention?", n_tokens=4)):
    print(repr(chunk[:120]))
    if i >= 3:
        break

In [ ]:
# Asyncio-based streaming for concurrent request handling.
#
# A production LLM server handles thousands of concurrent streaming requests.
# Blocking (synchronous) generators cannot serve concurrent requests efficiently
# because each request occupies a thread while waiting for the next token.
#
# Asyncio solves this with cooperative multitasking: when one coroutine is
# waiting for a token (sleeping), the event loop runs other coroutines.
# This allows a single thread to manage thousands of concurrent streams.
#
# The pattern below is what frameworks like FastAPI use under the hood when
# you return a StreamingResponse from an async endpoint.
#
# Note: we use asyncio.sleep(0) as a 'yield point' to allow other coroutines
# to run between tokens. Without this, a single long-running stream would
# block all other concurrent streams.

async def async_decode_stream(
    prompt:      str,
    n_tokens:    int   = 20,
    prefill_ms:  float = 100.0,
    decode_ms:   float = 25.0,
) -> AsyncIterator[str]:
    """
    Async generator for token streaming. Yields control between each token.

    In a real implementation, the GPU inference call would be an async
    operation (using CUDA streams or a background thread with asyncio.to_thread).
    Here we simulate with asyncio.sleep() which yields to the event loop.
    """
    vocab = ['neural', 'network', 'language', 'model', 'training',
             'inference', 'attention', 'token', 'embedding', 'layer']
    rng = np.random.default_rng(len(prompt))

    await asyncio.sleep(prefill_ms / 1000)  # simulate prefill

    for i in range(n_tokens):
        await asyncio.sleep(decode_ms / 1000)  # simulate decode step
        yield rng.choice(vocab)


async def stream_one_request(request_id: int, prompt: str, n_tokens: int):
    """Process one streaming request and return its timing."""    t_start = time.time()
    ttft    = None
    count   = 0
    async for token in async_decode_stream(prompt, n_tokens=n_tokens,
                                           prefill_ms=80, decode_ms=20):
        if ttft is None:
            ttft = time.time() - t_start
        count += 1
    return request_id, ttft, time.time() - t_start, count


async def concurrent_streaming_demo():
    """Run 4 requests concurrently and measure their timing."""    prompts = [
        ("What is a transformer?",  15),
        ("Explain backpropagation.", 10),
        ("What is softmax?",         8),
        ("Describe attention.",      12),
    ]
    tasks = [
        stream_one_request(i, prompt, n)
        for i, (prompt, n) in enumerate(prompts)
    ]
    results = await asyncio.gather(*tasks)
    return results


# Run the concurrent demo
results = asyncio.run(concurrent_streaming_demo())

print("Concurrent streaming (4 simultaneous requests):")
print(f"{'Req':<5} {'TTFT (ms)':>12} {'E2E (ms)':>12} {'Tokens':>8}")
print("-" * 42)
for req_id, ttft, e2e, n in results:
    print(f"{req_id:<5} {ttft*1000:>12.0f} {e2e*1000:>12.0f} {n:>8}")

print()
print("All 4 requests ran concurrently on the same thread.")
print("Sequential execution would take: sum of E2E times")
print(f"  Sequential: {sum(r[2] for r in results)*1000:.0f}ms")
print(f"  Concurrent: {max(r[2] for r in results)*1000:.0f}ms (wall clock)")

In [ ]:
# Stopping criteria and stream cancellation.
#
# Real streaming generators need several stopping conditions beyond simply
# reaching the max token count:
#
#   1. EOS token: the model generates a special end-of-sequence token
#   2. Stop strings: user-specified strings that terminate generation (e.g. '\n\n')
#   3. Token budget: hard limit to prevent runaway generation
#   4. Client cancellation: client disconnects mid-stream (HTTP connection drops)
#
# Stop string detection is tricky because a stop string may span multiple tokens.
# For example, '\n\n' might be tokenised as '\n' + '\n' across two steps.
# We must buffer the last len(stop_string) characters to detect multi-token matches.
#
# Client cancellation is handled differently in each framework:
#   - FastAPI: check request.is_disconnected() before yielding each chunk
#   - gRPC: catch the RpcError.CANCELLED exception
#   - WebSocket: catch the WebSocketDisconnect exception
# All approaches follow the same pattern: check a cancellation signal per token.

class StreamingDecoder:
    """
    Streaming decoder with stopping criteria and cancellation support.

    Wraps a token generator and applies stopping conditions at each step.
    This is the production-grade version of the simple decode_stream function.
    """

    def __init__(
        self,
        max_new_tokens:  int          = 200,
        stop_strings:    List[str]    = None,
        eos_token:       str          = '<|endoftext|>',
    ):
        self.max_new_tokens = max_new_tokens
        self.stop_strings   = stop_strings or []
        self.eos_token      = eos_token
        self._cancelled     = False

    def cancel(self):
        """Signal the stream to stop at the next token boundary."""        self._cancelled = True

    def stream(self, prompt: str, decode_ms: float = 20.0) -> Iterator[dict]:
        """
        Stream tokens with stopping criteria applied.

        Yields dicts with keys:
          token:        the generated token text
          token_count:  running count of generated tokens
          stop_reason:  None if generation continues, else the stop reason
        """
        vocab   = ['the', 'model', 'generates', 'tokens', 'language', 'neural',
                   'attention', 'training', 'gradient', 'loss', 'weight', 'layer',
                   'batch', 'softmax', 'embedding', 'transformer', 'encoder',
                   'decoder', 'sequence', 'output']
        rng     = np.random.default_rng(len(prompt))
        buffer  = ''          # rolling window for stop string detection
        n_generated = 0

        # Simulate prefill
        time.sleep(0.05)

        for _ in range(self.max_new_tokens):
            if self._cancelled:
                yield {'token': '', 'token_count': n_generated,
                       'stop_reason': 'cancelled'}
                return

            time.sleep(decode_ms / 1000)
            token = rng.choice(vocab)

            # Check for EOS
            if token == self.eos_token:
                yield {'token': '', 'token_count': n_generated, 'stop_reason': 'eos'}
                return

            n_generated += 1
            buffer      += token + ' '
            # Keep buffer to the length of the longest stop string + some slack
            max_stop_len = max((len(s) for s in self.stop_strings), default=0) + 10
            buffer       = buffer[-max_stop_len:]

            yield {'token': token, 'token_count': n_generated, 'stop_reason': None}

            # Check stop strings after yielding
            for stop in self.stop_strings:
                if stop in buffer:
                    yield {'token': '', 'token_count': n_generated, 'stop_reason': f'stop_string:{stop}'}
                    return

        yield {'token': '', 'token_count': n_generated, 'stop_reason': 'max_tokens'}


# Demo 1: Normal completion
print("Demo 1: Stream to max tokens")
decoder = StreamingDecoder(max_new_tokens=8)
tokens = []
for event in decoder.stream("Explain backpropagation"):
    if event['token']:
        tokens.append(event['token'])
    if event['stop_reason']:
        print(f"  Stopped: {event['stop_reason']}")
print(f"  Tokens: {' '.join(tokens)}")
print()

# Demo 2: Cancellation
print("Demo 2: Client cancellation after 3 tokens")
decoder = StreamingDecoder(max_new_tokens=20)
tokens = []
for i, event in enumerate(decoder.stream("Explain attention")):
    if event['token']:
        tokens.append(event['token'])
    if i == 2:
        decoder.cancel()
    if event['stop_reason']:
        print(f"  Stopped after {event['token_count']} tokens: {event['stop_reason']}")
print(f"  Tokens received: {' '.join(tokens)}")

In [ ]:
# Visualise TTFT, TPOT, and perceived quality.
#
# User experience research (Google, 2023) shows that perceived quality of
# LLM streaming responses correlates with:
#   1. TTFT < 500ms: users expect the first word in under half a second
#   2. TPOT < 50ms: streaming at >20 tokens/second feels natural (reading speed)
#   3. No pauses: irregular TPOT (jitter) is more disruptive than constant slow TPOT
#
# We simulate three configurations and visualise their token arrival timelines.
# This makes the difference between 'fast model' vs 'responsive model' concrete.

configs = [
    {'name': 'Slow TTFT, fast TPOT',  'prefill_ms': 800,  'decode_ms': 20,  'color': '#d62728'},
    {'name': 'Fast TTFT, slow TPOT',  'prefill_ms': 80,   'decode_ms': 80,  'color': '#ff7f0e'},
    {'name': 'Fast TTFT, fast TPOT',  'prefill_ms': 80,   'decode_ms': 25,  'color': '#2ca02c'},
]
N_TOKENS = 30

fig, axes = plt.subplots(len(configs), 1, figsize=(13, 8), sharex=True)

for ax, cfg in zip(axes, configs):
    token_times = [cfg['prefill_ms']]
    for i in range(1, N_TOKENS):
        token_times.append(token_times[-1] + cfg['decode_ms'])

    ax.plot(token_times, range(N_TOKENS), 'o-', color=cfg['color'],
            linewidth=2, markersize=4, label=cfg['name'])

    # Shade the prefill region
    ax.axvline(cfg['prefill_ms'], color=cfg['color'], linestyle='--', alpha=0.5)
    ax.text(cfg['prefill_ms'] + 5, N_TOKENS * 0.8, f"TTFT={cfg['prefill_ms']}ms",
            color=cfg['color'], fontsize=9)

    e2e = token_times[-1]
    tpot = cfg['decode_ms']
    ax.set_ylabel('Tokens received', fontsize=10)
    ax.set_title(f"{cfg['name']}  (TTFT={cfg['prefill_ms']}ms, TPOT={tpot}ms, E2E={e2e:.0f}ms)",
                 fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim(-1, N_TOKENS + 2)

axes[-1].set_xlabel('Time (ms)', fontsize=11)
plt.suptitle('Token Arrival Timelines: TTFT vs TPOT Trade-off', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/09_streaming_timelines.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key insight:")
print("  'Slow TTFT' model: user sees a blank screen for 800ms -- feels unresponsive")
print("  'Fast TTFT' model: user sees the first word in 80ms -- feels instant")
print("  Both finish in similar total time, but perceived quality differs dramatically.")
print()
print("Rule of thumb:")
print("  TTFT should be < 500ms for interactive applications")
print("  TPOT should be < 50ms (> 20 tokens/s) to match comfortable reading speed")

## 3. Engineering Notes

**Streaming production checklist**

| Concern | Recommendation |
|---------|---------------|
| TTFT optimisation | Chunked prefill, speculative decoding for first token, prompt caching |
| TPOT consistency | Avoid batch size variation during a stream; use dedicated decode instances |
| Stop string detection | Buffer last `max(stop_string_lengths)` characters; check after each token |
| Client disconnection | Poll `request.is_disconnected()` per token; free KV cache immediately on cancel |
| Token budget | Always enforce `max_new_tokens`; never let generation run indefinitely |
| Backpressure | If the client is consuming tokens slower than generation, buffer or slow down |

## 4. Exercises

1. **TTFT optimisation with prompt caching**: Implement a simple prompt prefix cache. If two requests share the first 64 tokens, reuse the computed KV cache. Measure the TTFT reduction.

2. **SSE client implementation**: Write a Python client that consumes the SSE stream from `streaming_response()`, assembles the tokens into a string, and measures TTFT and TPOT.

3. **Jitter simulation**: Modify `decode_stream` to add random jitter to decode time (uniform noise ±20ms). Measure P95 TPOT with and without jitter. Does jitter affect perceived quality more than mean TPOT?

4. **Multi-modal stopping**: Extend `StreamingDecoder` to support stop after a JSON object is complete (count `{` and `}` to detect balanced JSON). Test with a prompt that asks the model to respond in JSON format.

5. **Streaming benchmarking**: Use the `aiohttp` library to make 10 concurrent streaming requests to a local vLLM server (or use the mock from this notebook). Plot the CDF of TTFT values. Compare against a non-streaming endpoint.

---

## 5. References

- [OpenAI API Streaming Documentation](https://platform.openai.com/docs/api-reference/streaming)
- [Anthropic Streaming Documentation](https://docs.anthropic.com/en/api/messages-streaming)
- [FastAPI StreamingResponse](https://fastapi.tiangolo.com/advanced/custom-response/#streamingresponse)
- [HuggingFace TextIteratorStreamer](https://huggingface.co/docs/transformers/internal/generation_utils#transformers.TextIteratorStreamer)